# LLM-Assisted Transaction Reason-Code Classifier

**Business Problem:**  
In a high-velocity F&B transaction environment (400–600 orders/day), exception triage requires manually labeling each discrepancy with a standardized reason code. This notebook automates that classification step using an LLM with confidence scoring and Dead Letter Queue routing.

**Pipeline:**
1. Simulate 1,000 transaction records with intentional noise
2. Schema validation → route malformed records to Dead Letter Queue
3. LLM classifies valid records with reason code + confidence score
4. Confidence threshold (0.7): below → human review queue; above → auto-approved

**Reason Code Taxonomy:**
- `Weekend Batch Delay` — Settlement timing gap due to weekend/holiday batching
- `Gateway Timeout` — Payment gateway failure or connection drop
- `Duplicate Entry` — Same transaction appears more than once in settlement files
- `Refund Mismatch` — Refund processed but not reconciled in ledger
- `Fee Deduction Gap` — Platform fee discrepancy not captured in revenue report
- `Unclassified` — LLM cannot confidently assign a reason code

In [ ]:
# Step 1: Install dependencies
!pip install openai pandas -q

In [ ]:
# Step 2: Set your OpenAI API Key
import os
os.environ['OPENAI_API_KEY'] = 'sk-YOUR-API-KEY-HERE'  # Replace with your key

In [ ]:
# Step 3: Simulate 1,000 transaction records with intentional noise
import pandas as pd
import random
import numpy as np

random.seed(42)
np.random.seed(42)

TEMPLATES = [
    ("Settlement amount lower than expected; deposit received 48 hours after transaction date", "Weekend Batch Delay"),
    ("Order marked complete but payment not reflected in ledger after 2 hours", "Gateway Timeout"),
    ("Same transaction ID appears twice in daily reconciliation report", "Duplicate Entry"),
    ("Refund issued but original charge still showing as outstanding", "Refund Mismatch"),
    ("Deposit amount does not match; difference traced to platform fee deduction not reflected in report", "Fee Deduction Gap"),
    ("Payment gateway returned error code 504; transaction status unknown", "Gateway Timeout"),
    ("Weekend batch grouped Monday deposits with Sunday transactions causing timing mismatch", "Weekend Batch Delay"),
    ("Customer disputed charge; refund processed but ledger not updated", "Refund Mismatch"),
    ("Transaction ID duplicated across two settlement files from same day", "Duplicate Entry"),
    ("Platform fee rate changed mid-month; discrepancy between expected and actual deduction", "Fee Deduction Gap"),
]

records = []
for i in range(1000):
    txn_id = f'TXN{i+1:04d}'
    # Inject ~33% noise: missing fields, null amounts, malformed IDs
    noise_type = random.random()
    if noise_type < 0.10:
        # Missing description
        records.append({'transaction_id': txn_id, 'amount': round(random.uniform(10, 600), 2),
                         'platform': random.choice(['Meituan','Douyin']), 'description': None, 'manual_label': None})
    elif noise_type < 0.20:
        # Null amount
        desc, label = random.choice(TEMPLATES)
        records.append({'transaction_id': txn_id, 'amount': None,
                         'platform': random.choice(['Meituan','Douyin']), 'description': desc, 'manual_label': label})
    elif noise_type < 0.33:
        # Malformed transaction ID
        records.append({'transaction_id': f'BAD-{i}', 'amount': round(random.uniform(10, 600), 2),
                         'platform': random.choice(['Meituan','Douyin']), 'description': 'unknown error', 'manual_label': None})
    else:
        desc, label = random.choice(TEMPLATES)
        records.append({'transaction_id': txn_id, 'amount': round(random.uniform(10, 600), 2),
                         'platform': random.choice(['Meituan','Douyin']), 'description': desc, 'manual_label': label})

raw_df = pd.DataFrame(records)
print(f'Simulated {len(raw_df)} raw transaction records')
raw_df.head()

In [ ]:
# Step 4: Schema Validation → Dead Letter Queue
def validate_record(row):
    """Returns True if record passes schema validation."""
    if pd.isna(row['description']) or str(row['description']).strip() == '':
        return False, 'missing_description'
    if pd.isna(row['amount']):
        return False, 'null_amount'
    if not str(row['transaction_id']).startswith('TXN'):
        return False, 'malformed_id'
    return True, None

valid_rows = []
dlq_rows = []

for _, row in raw_df.iterrows():
    is_valid, reason = validate_record(row)
    if is_valid:
        valid_rows.append(row)
    else:
        row_copy = row.copy()
        row_copy['dlq_reason'] = reason
        dlq_rows.append(row_copy)

valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
dlq_df = pd.DataFrame(dlq_rows).reset_index(drop=True)

print(f'Schema Validation Results:')
print(f'  Valid records:       {len(valid_df)}')
print(f'  Dead Letter Queue:   {len(dlq_df)}')
print(f'  DLQ breakdown:')
print(dlq_df['dlq_reason'].value_counts().to_string())

In [ ]:
# Step 5: Define LLM Classifier with Confidence Score
import json
from openai import OpenAI

client = OpenAI()

REASON_CODES = [
    'Weekend Batch Delay', 'Gateway Timeout', 'Duplicate Entry',
    'Refund Mismatch', 'Fee Deduction Gap', 'Unclassified'
]

CONFIDENCE_THRESHOLD = 0.7

SYSTEM_PROMPT = """You are a financial operations analyst specializing in payment reconciliation.
Classify transaction discrepancy descriptions into standardized reason codes.

Available reason codes:
- Weekend Batch Delay: Settlement timing gap due to weekend or holiday batch processing
- Gateway Timeout: Payment gateway failure, connection error, or timeout
- Duplicate Entry: Same transaction appears more than once in settlement files
- Refund Mismatch: Refund processed but not correctly reconciled in the ledger
- Fee Deduction Gap: Platform fee or deduction discrepancy not captured in revenue report
- Unclassified: Cannot confidently assign any of the above codes

Respond ONLY with valid JSON in this exact format:
{"reason_code": "<code>", "confidence": <0.0-1.0>, "reasoning": "<one sentence>"}"""

def classify_transaction(description):
    """Classify a transaction description. Returns (reason_code, confidence, routing)."""
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify: {description}"}
            ],
            max_tokens=80,
            temperature=0
        )
        raw = response.choices[0].message.content.strip()
        result = json.loads(raw)
        reason_code = result.get('reason_code', 'Unclassified')
        confidence = float(result.get('confidence', 0.0))
        
        # Validate reason code
        if reason_code not in REASON_CODES:
            reason_code = 'Unclassified'
            confidence = 0.0
        
        # Confidence threshold routing
        if confidence >= CONFIDENCE_THRESHOLD:
            routing = 'Auto-Approved'
        else:
            routing = 'Human Review Queue'
        
        return reason_code, confidence, routing, result.get('reasoning', '')
    
    except Exception as e:
        # Graceful degradation: API failure → human review, pipeline does not crash
        return 'Unclassified', 0.0, 'Human Review Queue', f'API error: {e}'

print(f'Classifier defined. Confidence threshold: {CONFIDENCE_THRESHOLD}')

In [ ]:
# Step 6: Test with one record
test_desc = valid_df['description'].iloc[0]
code, conf, routing, reasoning = classify_transaction(test_desc)

print(f'Description: {test_desc}')
print(f'Reason Code: {code}')
print(f'Confidence:  {conf}')
print(f'Routing:     {routing}')
print(f'Reasoning:   {reasoning}')

In [ ]:
# Step 7: Run classifier on all valid records
# NOTE: Running on full 664 records costs ~$0.05-0.10
# For demo, sample 50 records; remove [:50] to run full dataset
import time

sample_df = valid_df.copy()  # Remove [:50] to classify all valid records

reason_codes, confidences, routings, reasonings = [], [], [], []

for i, row in sample_df.iterrows():
    code, conf, routing, reasoning = classify_transaction(row['description'])
    reason_codes.append(code)
    confidences.append(conf)
    routings.append(routing)
    reasonings.append(reasoning)
    if (i+1) % 50 == 0:
        print(f'  Processed {i+1}/{len(sample_df)}...')
    time.sleep(0.3)

sample_df = sample_df.copy()
sample_df['llm_reason_code'] = reason_codes
sample_df['confidence'] = confidences
sample_df['routing'] = routings
sample_df['reasoning'] = reasonings

print(f'\nClassification complete.')

In [ ]:
# Step 8: Results Summary
auto = sample_df[sample_df['routing'] == 'Auto-Approved']
review = sample_df[sample_df['routing'] == 'Human Review Queue']

print('=== Pipeline Summary ===')
print(f'Total raw records:       1000')
print(f'Dead Letter Queue:       {len(dlq_df)}')
print(f'Valid (classified):      {len(sample_df)}')
print(f'  Auto-Approved (≥0.7):  {len(auto)}')
print(f'  Human Review (<0.7):   {len(review)}')
print()
print('=== Reason Code Distribution ===')
print(sample_df['llm_reason_code'].value_counts().to_string())
print()
print('=== Avg Confidence by Reason Code ===')
print(sample_df.groupby('llm_reason_code')['confidence'].mean().round(2).to_string())

In [ ]:
# Step 9: Generate QA Exception Report
output = sample_df[['transaction_id', 'amount', 'platform', 'description',
                     'manual_label', 'llm_reason_code', 'confidence', 'routing', 'reasoning']].copy()

output.to_csv('output_classified.csv', index=False)
print('QA Exception Report saved: output_classified.csv')
output.head(10)

## Results Summary

| Stage | Count |
|---|---|
| Raw records simulated | 1,000 |
| Dead Letter Queue (schema failures) | ~336 |
| Valid records classified | ~664 |
| Auto-Approved (confidence ≥ 0.7) | ~600 |
| Routed to Human Review (< 0.7) | ~64 |

**Design Decisions:**
- `temperature=0`: Deterministic, consistent output
- **Confidence threshold 0.7**: Low-confidence results route to human review — reduces false labeling
- **Dead Letter Queue**: Schema failures isolated before LLM call — clean inputs, lower cost
- **Graceful degradation**: API errors route to human review, pipeline never crashes
- **JSON response format**: Structured output enables downstream automation

**Connection to IEEE Fraud Monitoring Project:**  
This classifier extends the [End-to-End Fraud Monitoring Pipeline](https://github.com/danielyin89/End-to-End-Fraud-Monitoring-Risk-Strategy-Pipeline) by automating the reason-code labeling step that previously required manual triage.